In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movie_cast_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_cast.json")
    )

castOrder,characterName,genderId,movieId,personId
0,Captain Jack Sparrow,2,285,85
1,Will Turner,2,285,114
2,Elizabeth Swann,1,285,116
3,William Bootstrap Bill Turner,2,285,1640
4,Captain Sao Feng,2,285,1619
5,Captain Davy Jones,2,285,2440
6,Captain Hector Barbossa,2,285,118
7,Admiral James Norrington,2,285,1709
8,Joshamee Gibbs,2,285,2449
9,Lord Cutler Beckett,2,285,2441


In [0]:
movie_cast_schema = StructType([
    StructField("castOrder", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True)
])
movie_cast_df = (
    spark.read
        .schema(movie_cast_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/incremental_append/{v_file_date}/movie_cast.json")
)

In [0]:
movie_cast_df = (
    add_ingestion_date(movie_cast_df)
        .select(
            col("movieId").alias("movie_id"),
            col("personId").alias("person_id"),
            col("characterName").alias("character_name"),
            lit(v_env).alias("env"),
            lit(v_file_date).alias("file_date")
        )
)

In [0]:
#overwrite_partitioned_table("movie_silver", "movie_cast", "file_date", v_file_date)

In [0]:
"""
(
    movie_cast_df
        .write
            .mode("append")
            .partitionBy("file_date")
            .saveAsTable("movie_silver.movie_cast")
)
"""

In [0]:
merge_condition = 'trgt.movie_id = src.movie_id AND trgt.person_id = src.person_id AND trgt.file_date = src.file_date'
merge_delta_lake(input_df = movie_cast_df, schema_name = "movie_silver", table_name = "movie_cast", merge_condition = merge_condition, partition_column = "file_date")

In [0]:
dbutils.notebook.exit("Success")